# Entity graph: people, organizations, topics

Interactive network from the speakers data: **green** nodes are topics/projects,
**blue** are people, **orange** are organizations. Edges connect people to
topics they addressed (red = spoke against, green = for, gray = neutral/mixed)
and to their stated affiliations. Drag, zoom, hover for details.

Tune `TOPIC_RULES`, `MIN_PERSON`, `MIN_ORG` and re-run.

In [1]:
import duckdb
import pandas as pd
from collections import Counter, defaultdict
from pyvis.network import Network

con = duckdb.connect()
for t in ("meetings", "speakers"):
    con.execute(f"CREATE VIEW {t} AS SELECT * FROM '../data/db/{t}.parquet'")

sp = con.execute("""
    SELECT s.name, s.affiliation, s.topic, s.position, m.date::VARCHAR AS date
    FROM speakers s JOIN meetings m USING (meeting_id)
    WHERE s.name IS NOT NULL
""").df()
print(len(sp), "named speaker records")

1351 named speaker records


In [2]:
# keyword -> topic-entity mapping (first match wins; extend freely)
TOPIC_RULES = [
    ("Monitor Point", ["monitor point", "40 quay", "56 quay"]),
    ("Bushwick Inlet Park", ["bushwick inlet"]),
    ("River Ring", ["river ring"]),
    ("TAO / 11-25 Franklin", ["11-25 franklin", "tao group", "franklin bk"]),
    ("Nightlife / SLA", ["liquor", "sla ", "nightclub", "nightlife", "bar ", "applicant"]),
    ("National Grid / Newtown Creek", ["national grid", "newtown creek", "pipeline", "superfund"]),
    ("Cannabis", ["cannabis", "dispensary", "smoke shop"]),
    ("Parks (other)", ["park", "playground", "open space"]),
    ("Streets / transport", ["street", "bike", "traffic", "bus ", "g train", "bqe", "mcguinness"]),
    ("Housing / land use", ["rezon", "housing", "development", "ulurp", "landmark"]),
]

def classify(topic):
    low = (topic or "").lower()
    for label, keys in TOPIC_RULES:
        if any(k in low for k in keys):
            return label
    return None

sp["entity"] = sp.topic.map(classify)
sp["aff"] = (sp.affiliation.str.strip().str.title()
             .replace({"Resident": None, "Residents": None}))  # 'resident' isn't an org
classified = sp.dropna(subset=["entity"])
print(f"{len(classified)} records mapped to {classified.entity.nunique()} topic entities")

855 records mapped to 10 topic entities


In [3]:
MIN_PERSON = 2   # min classified speeches for a person node
MIN_ORG = 2      # min records for an org node
POS_COLOR = {"against": "#c0392b", "for": "#27ae60"}

def build_graph(df, height="750px"):
    net = Network(height=height, width="100%", notebook=True,
                  cdn_resources="in_line", bgcolor="#ffffff")
    people = df.groupby("name").size()
    people = people[people >= MIN_PERSON].index

    for ent, n in df.entity.value_counts().items():
        net.add_node(f"T:{ent}", label=ent, color="#2d6a4f", shape="dot",
                     size=14 + min(n, 60) * 0.6, title=f"{n} speeches")

    pdf = df[df.name.isin(people)]
    for name, grp in pdf.groupby("name"):
        span = f"{grp.date.min()[:4]}–{grp.date.max()[:4]}"
        net.add_node(f"P:{name}", label=name, color="#2b6cb0", shape="dot",
                     size=8 + len(grp), title=f"{len(grp)} speeches, {span}")
        for ent, egrp in grp.groupby("entity"):
            pos = Counter(egrp.position)
            top, _ = pos.most_common(1)[0]
            color = POS_COLOR.get(top, "#95a5a6")
            sample = "; ".join(egrp.topic.head(3).str[:60])
            net.add_edge(f"P:{name}", f"T:{ent}", value=len(egrp),
                         color=color, title=f"{dict(pos)}\n{sample}")

    orgs = pdf.dropna(subset=["aff"]).groupby("aff").size()
    for org in orgs[orgs >= MIN_ORG].index:
        net.add_node(f"O:{org}", label=org, color="#e67e22", shape="box",
                     title="organization")
        for name in pdf[pdf.aff == org].name.unique():
            if f"P:{name}" in net.get_nodes():
                net.add_edge(f"O:{org}", f"P:{name}", color="#f0c8a0", dashes=True)

    net.barnes_hut(gravity=-8000, central_gravity=0.4, spring_length=120)
    return net

net = build_graph(classified)
print(f"{len(net.get_nodes())} nodes, {len(net.get_edges())} edges")
net.show("entity_graph.html")

96 nodes, 136 edges
entity_graph.html


Focused view — waterfront saga only:

In [4]:
WATERFRONT = ["Monitor Point", "Bushwick Inlet Park", "River Ring",
              "TAO / 11-25 Franklin", "National Grid / Newtown Creek"]
net2 = build_graph(classified[classified.entity.isin(WATERFRONT)], height="650px")
net2.show("waterfront_graph.html")

waterfront_graph.html
